### Writing to sentinel-5P zarr store

Necessary imports

In [ ]:
import zarr 
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from scipy.interpolate import griddata
from pyproj import Transformer, CRS
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject, Resampling

import os

ModuleNotFoundError: No module named 'reprojection'

In [ ]:
product_type = "CO"
store_path = "/home/simon/eodc_datasync/private/eodc_logs/s5p/s5p.zarr"


store = zarr.storage.LocalStore(store_path)
group = zarr.group(store=store, path=product_type)

For now we will only write CO data to the zarr store. For this we will go through the folder containing our netCDF files and reproject them to EUQI7 grid, as some files don't extent over Austria a potential error is overriden.

In [36]:
qa_value = 0.5
is_no2 = True

In [37]:
path_base = "/home/simon/eodc_datasync/products/copernicus.eu/.incoming/s5p/"

In [34]:
ds = xr.open_dataset("/home/simon/eodc_datasync/products/copernicus.eu/.incoming/s5p/S5P_OFFL_L2__NO2____20241204T102958_20241204T121128_37016_03_020800_20241206T081617.nc", engine="netcdf4", group = "PRODUCT")
ds

<xarray.Dataset> Size: 331MB
Dimensions:                                               (time: 1,
                                                           scanline: 4172,
                                                           ground_pixel: 450,
                                                           layer: 34,
                                                           vertices: 2,
                                                           corner: 4,
                                                           polynomial_exponents: 6,
                                                           intensity_offset_polynomial_exponents: 1)
Coordinates:
  * scanline                                              (scanline) float64 33kB ...
  * ground_pixel                                          (ground_pixel) float64 4kB ...
  * time                                                  (time) datetime64[ns] 8B ...
  * corner                                                (corner) float64 32B ...
  * polynomial_exponents                                  (polynomial_exponents) float64 48B ...
  * intensity_offset_polynomial_exponents                 (intensity_offset_polynomial_exponents) float64 8B ...
  * layer                                                 (layer) float64 272B ...
  * vertices                                              (vertices) float64 16B ...
    latitude                                              (time, scanline, ground_pixel) float32 8MB ...
    longitude                                             (time, scanline, ground_pixel) float32 8MB ...
Data variables:
    delta_time                                            (time, scanline) datetime64[ns] 33kB ...
    time_utc                                              (time, scanline) object 33kB ...
    qa_value                                              (time, scanline, ground_pixel) float32 8MB ...
    nitrogendioxide_tropospheric_column                   (time, scanline, ground_pixel) float32 8MB ...
    nitrogendioxide_tropospheric_column_precision         (time, scanline, ground_pixel) float32 8MB ...
    nitrogendioxide_tropospheric_column_precision_kernel  (time, scanline, ground_pixel) float32 8MB ...
    averaging_kernel                                      (time, scanline, ground_pixel, layer) float32 255MB ...
    air_mass_factor_troposphere                           (time, scanline, ground_pixel) float32 8MB ...
    air_mass_factor_total                                 (time, scanline, ground_pixel) float32 8MB ...
    tm5_tropopause_layer_index                            (time, scanline, ground_pixel) float64 15MB ...
    tm5_constant_a                                        (layer, vertices) float32 272B ...
    tm5_constant_b                                        (layer, vertices) float32 272B ...

In [ ]:
datasets = []
for file in os.listdir(path_base):
        print(file)
        path = os.path.join(path_base, file)
        print(path)
        if file.startswith("S5P_OFFL"):
            try:
                datasets.append(reprojection.reproject_data(path, qa_value, is_no2))
            except (ValueError) as e:
                print(f"Warning file {file}: {e}")
                continue
        else:
            continue

S5P_OFFL_L2__NO2____20240428T092517_20240428T110647_33894_03_020600_20240501T103229.nc
/home/simon/eodc_datasync/products/copernicus.eu/.incoming/s5p/S5P_OFFL_L2__NO2____20240428T092517_20240428T110647_33894_03_020600_20240501T103229.nc
S5P_OFFL_L2__NO2____20241204T102958_20241204T121128_37016_03_020800_20241206T081617.nc
/home/simon/eodc_datasync/products/copernicus.eu/.incoming/s5p/S5P_OFFL_L2__NO2____20241204T102958_20241204T121128_37016_03_020800_20241206T081617.nc
S5P_OFFL_L2__NO2____20240129T104405_20240129T122535_32618_03_020600_20240201T110129.nc
/home/simon/eodc_datasync/products/copernicus.eu/.incoming/s5p/S5P_OFFL_L2__NO2____20240129T104405_20240129T122535_32618_03_020600_20240201T110129.nc
S5P_OFFL_L2__NO2____20240501T100956_20240501T115126_33937_03_020600_20240504T082421.nc
/home/simon/eodc_datasync/products/copernicus.eu/.incoming/s5p/S5P_OFFL_L2__NO2____20240501T100956_20240501T115126_33937_03_020600_20240504T082421.nc
S5P_OFFL_L2__NO2____20240203T090918_20240203T105048_

The datasets in the datasets list are now concatenated, if the timecoordinate is the same for two datasets the data is averaged.

In [ ]:
merged = reprojection.merge_mean_by_time(datasets)

Now we can open the zarr store

In [ ]:
product_type = "NO2"
store_path = "/eodc/private/eodc_logs/s5p/s5p.zarr"


store = zarr.storage.LocalStore(store_path)
group = zarr.group(store=store, path=product_type)

The time indexes are then calculated

In [ ]:
time_origin = np.datetime64("2018-04-01")
time_min = (merged.time.min().values.astype("datetime64[D]") - time_origin).astype("int64")
time_max = (merged.time.max().values.astype("datetime64[D]") - time_origin).astype("int64")

Next the data is written to the correct position in the zarr store. The correct nodata value and scale factor is applied before writing

In [ ]:
for var in merged.data_vars:
    scale = group[var].attrs["scale_factor"]
    fill_value = group[var].attrs["_FillValue"]
    data = np.nan_to_num(np.round(group[var].values/scale), nan=fill_value)
    group[var][time_min:time_max, :, :] = data